In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# Phase-Amplitude Coupling Analysis

This notebook computes phase-amplitude coupling (PAC) for ACC and dmPFC electrodes,
performs electrode-level cluster permutation testing, and generates cluster-masked
comodulogram visualizations.

## 1. Setup: Imports and Paths

In [ ]:
import os
import numpy as np
import mne
from tensorpac import Pac
from mne.stats import permutation_cluster_1samp_test
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# Set plotting style
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10


In [ ]:
# Define all paths
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
ephys_dir = f'{base_dir}/projects/guLab/Salman/EphysAnalyses'

# Output directory - new scratch location
save_dir = '/Volumes/T7 Shield/scratch/MemoryBandit/pac'
save_dir_data = f'{save_dir}/data'
save_dir_plots = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-PAC'

# Create output directories if they don't exist
os.makedirs(save_dir_data, exist_ok=True)
os.makedirs(save_dir_plots, exist_ok=True)

print(f"Data will be saved to: {save_dir_data}")
print(f"Plots will be saved to: {save_dir_plots}")


In [ ]:
# Load demographics and filter subjects
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)]

# Exclude problematic subjects
excluded_subjects = ['MS015', 'MS024', 'MS034', 'MS038']
subj_df = subj_df[~subj_df['MSSMCode'].isin(excluded_subjects)]

subj_ids = subj_df.MSSMCode.values
print(f"Number of subjects: {len(subj_ids)}")
print(f"Subjects: {subj_ids}")

## 2. Configuration: Frequency Bands and PAC Parameters


In [ ]:
# Phase frequency bands (3-30 Hz)
f_pha_start, f_pha_end = 4, 30
f_pha_nb = 20
f_pha_mid = np.linspace(f_pha_start, f_pha_end, f_pha_nb)
f_pha = np.c_[f_pha_mid - f_pha_mid / 4., f_pha_mid + f_pha_mid / 4.]

# Amplitude frequency bands (70-200 Hz - HFA)
f_amp_start, f_amp_end = 70, 200
f_amp_nb = 10
f_amp_mid = np.linspace(f_amp_start, f_amp_end, f_amp_nb)
f_amp = np.c_[f_amp_mid - f_amp_mid / 8., f_amp_mid + f_amp_mid / 8.]

# PAC parameters
n_perm = 200  # Number of surrogates for PAC normalization
n_perm_cluster = 1000  # Number of permutations for cluster testing
cluster_alpha = 0.05  # Significance threshold for clusters

# Regions of interest
rois = ['ACC', 'HPC']

# Event configuration
event = 'feedback_start'
day = 'Day1'
buf_ms = 1000  # Buffer in ms to remove edge artifacts

print(f"Phase frequencies: {f_pha_start}-{f_pha_end} Hz ({f_pha_nb} bands)")
print(f"Amplitude frequencies: {f_amp_start}-{f_amp_end} Hz ({f_amp_nb} bands)")
print(f"ROIs: {rois}")


In [ ]:
# Initialize ERPAC object (Event-Related Phase-Amplitude Coupling)
pac_obj = Pac(idpac=(6, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')



## 3. Data Loading: Behavioral Data


In [ ]:
# Load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')

# Filter fast reaction times
# learn_df = learn_df[learn_df.gamble_rt > 0.3]

# Filter slow reaction times (>95th percentile per subject)
# for subj in learn_df.participant.unique():
#     rt_threshold = np.percentile(learn_df[learn_df.participant == subj].gamble_rt, 95)
#     learn_df.loc[(learn_df.participant == subj) & (learn_df.gamble_rt > rt_threshold), 'gamble_rt'] = np.nan

# learn_df.dropna(subset=['gamble_rt'], inplace=True)
print(f"Behavioral data loaded: {len(learn_df)} trials")


In [ ]:
# Create RPE_Category column using tercile split per subject
# This matches the definition in HFA_RPE_RSA.ipynb
if 'rpe_category' not in learn_df.columns:
    learn_df['rpe_category'] = np.nan
    
    for subj_id in learn_df.participant.unique():
        subj_mask = learn_df.participant == subj_id
        subj_rpe = learn_df.loc[subj_mask, 'rpe'].values
        
        # Compute terciles (33.33rd and 66.67th percentiles)
        terciles = np.percentile(subj_rpe, [33.33, 66.67])
        
        # Assign categories
        rpe_category = []
        for rpe in subj_rpe:
            if rpe <= terciles[0]:
                rpe_category.append('negative')
            elif rpe <= terciles[1]:
                rpe_category.append('neutral')
            else:
                rpe_category.append('pos')
        
        learn_df.loc[subj_mask, 'rpe_category'] = rpe_category
    
    print(f"RPE categories created using tercile split")
    print(f"Category distribution:")
    print(learn_df['rpe_category'].value_counts())
else:
    print(f"RPE category column already exists in learn_df")
    print(f"Category distribution:")
    print(learn_df['rpe_category'].value_counts())

## 4. PAC Computation


In [ ]:
# # Main PAC computation loop
# all_results = []

# for subj_id in tqdm(subj_ids, desc="Processing subjects"):
#     save_path = f'{ephys_dir}/{subj_id}/neural/{day}'
    
#     # Check if epoch file exists
#     epoch_file = f'{save_path}/{event}-epo.fif'
#     if not os.path.exists(epoch_file):
#         print(f"Skipping {subj_id}: epoch file not found")
#         continue
    
#     # Load epochs once per subject
#     epochs = mne.read_epochs(epoch_file, preload=True, verbose=False)
#     sfreq = epochs.info['sfreq']
#     edges = int(buf_ms * (sfreq / 1000))  # Convert buffer to samples
    
#     # Load behavioral data for this subject
#     subj_learn_df = learn_df[learn_df.participant == subj_id].copy()
    
#     # Ensure epochs and learn_df are aligned
#     # Assuming epochs are in the same order as learn_df for this subject
#     if len(epochs) != len(subj_learn_df):
#         print(f"Warning: {subj_id} has {len(epochs)} epochs but {len(subj_learn_df)} trials in learn_df")
#         # Take the minimum to ensure alignment
#         min_len = min(len(epochs), len(subj_learn_df))
#         epochs = epochs[:min_len]
#         subj_learn_df = subj_learn_df.head(min_len).reset_index(drop=True)
    
#     # Load electrode dataframe
#     elec_df = pd.read_csv(f'{ephys_dir}/{subj_id}/Day1_reref_elec_df')
    
#     # Filter to ROI electrodes
#     roi_elecs = elec_df[elec_df.salman_region.isin(rois)]
    
#     if len(roi_elecs) == 0:
#         print(f"Skipping {subj_id}: no electrodes in ROIs")
#         del epochs
#         continue
    
#     # Process each electrode in ROIs
#     for idx, row in roi_elecs.iterrows():
#         ch_name = row['label']
#         ch_region = row['salman_region']
        
#         # Check if channel exists in epochs
#         if ch_name not in epochs.ch_names:
#             continue
        
#         # Extract signal for this electrode: (n_trials, n_times)
#         sig = np.squeeze(epochs.copy().pick(ch_name)._data)
        
#         # Skip if too few trials
#         if len(sig) < 5:
#             print(f"Skipping {subj_id} {ch_name}: only {len(sig)} trials")
#             continue
        
#         # Compute PAC with Gaussian Copula method and surrogates
#         # For local PAC (same electrode), pass signal to both x_pha and x_amp
#         pac = pac_obj.filterfit(
#             sfreq, sig, x_amp=sig,
#             n_perm=n_perm,
#             n_jobs=-1,
#             edges=edges,
#             random_state=42
#         )
        
#         # Z-score the PAC values against surrogates
#         surr_mean = pac_obj.surrogates.mean(axis=0)
#         surr_std = pac_obj.surrogates.std(axis=0)
#         # Avoid division by zero
#         surr_std = np.where(surr_std == 0, 1, surr_std)
#         z_scored_pac = (pac - surr_mean) / surr_std
        
#         # Save z-scored PAC as compressed numpy
#         np.savez_compressed(
#             f'{save_dir_data}/{subj_id}_{ch_name}_pac.npz',
#             z_scored_pac=z_scored_pac,
#             f_pha=f_pha,
#             f_amp=f_amp,
#             region=ch_region,
#             n_trials=len(sig)
#         )
        
#         # Store metadata for results
#         all_results.append({
#             'subject': subj_id,
#             'electrode': ch_name,
#             'region': ch_region,
#             'n_trials': len(sig),
#             'hemisphere': row.get('hemisphere', 'unknown')
#         })
    
#     # Free memory
#     del epochs

# # Create results dataframe
# results_df = pd.DataFrame(all_results)
# print(f"\nPAC computed for {len(results_df)} electrode combinations")
# print(f"\nBreakdown by region:")
# print(results_df.groupby('region').size())


In [ ]:
# Compute PAC for all subjects and all electrodes in ROIs
# Save trial-level results for later analysis with learn_df

from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Initialize list to store all trial-level dataframes
all_trial_dfs = []

# Process each subject
for subj_id in tqdm(subj_ids, desc="Processing subjects"):
    print(f"\n{'='*60}")
    print(f"Processing subject: {subj_id}")
    print(f"{'='*60}")
    
    # Load epochs for this subject
    save_path = f'{ephys_dir}/{subj_id}/neural/{day}'
    epoch_file = f'{save_path}/{event}-epo.fif'
    
    if not os.path.exists(epoch_file):
        print(f"  Warning: Epoch file not found for {subj_id}, skipping...")
        continue
    
    # Load epochs
    try:
        epochs = mne.read_epochs(epoch_file, preload=True, verbose=False)
        sfreq = epochs.info['sfreq']
        edges = int(buf_ms * (sfreq / 1000))
        n_trials = len(epochs)
        print(f"  Loaded {n_trials} trials")
    except Exception as e:
        print(f"  Error loading epochs for {subj_id}: {e}")
        continue
    
    # Load electrode dataframe
    try:
        elec_df = pd.read_csv(f'{ephys_dir}/{subj_id}/Day1_reref_elec_df')
    except Exception as e:
        print(f"  Error loading electrode dataframe for {subj_id}: {e}")
        del epochs
        continue
    
    # Filter to ROI electrodes
    roi_elecs = elec_df[elec_df.salman_region.isin(rois)]
    
    if len(roi_elecs) == 0:
        print(f"  Warning: No electrodes in ROIs for {subj_id}, skipping...")
        del epochs
        continue
    
    print(f"  Found {len(roi_elecs)} electrodes in ROIs")
    
    # Get subject's learn_df to align trials
    subj_learn_df = learn_df[learn_df.participant == subj_id].copy()
    if len(subj_learn_df) == 0:
        print(f"  Warning: No behavioral data found for {subj_id} in learn_df")
        del epochs
        continue
    
    # Process each electrode in ROIs
    subj_trial_dfs = []
    
    for elec_idx, electrode in roi_elecs.iterrows():
        ch_name = electrode['label']
        ch_region = electrode['salman_region']
        
        # Check if channel exists
        if ch_name not in epochs.ch_names:
            print(f"    Skipping {ch_name}: not found in epochs")
            continue
        
        print(f"    Processing {ch_name} ({ch_region})...")
        
        try:
            # Extract signal for this electrode: (n_trials, n_times)
            sig = np.squeeze(epochs.copy().pick(ch_name)._data)
            
            # Compute PAC with Gaussian Copula method and surrogates
            pac = pac_obj.filterfit(sfreq, 
                                    sig, 
                                    n_perm=n_perm, n_jobs=-1,
                                    edges=edges, random_state=0)
            
            # Z-score against surrogates
            surr_mean = pac_obj.surrogates.mean(axis=0)
            surr_std = pac_obj.surrogates.std(axis=0)
            z_scored_pac = (pac - surr_mean) / surr_std
            
            # PAC shape is (n_amp_freqs, n_phase_freqs, n_trials) = (10, 20, n_trials)
            # Ensure we have the expected shape
            if z_scored_pac.ndim != 3:
                print(f"      Warning: Unexpected PAC shape {z_scored_pac.shape}, skipping...")
                continue
            
            n_amp_freqs, n_phase_freqs, n_trials_pac = z_scored_pac.shape
            
            # Align with learn_df - take minimum length
            min_trials = min(n_trials_pac, len(subj_learn_df))
            z_scored_pac = z_scored_pac[:, :, :min_trials]
            
            # Save full z-scored PAC array as .npz
            pac_save_file = f'{save_dir_data}/{subj_id}_{ch_name}_pac.npz'
            np.savez_compressed(
                pac_save_file,
                z_scored_pac=z_scored_pac,
                f_pha=f_pha_mid,
                f_amp=f_amp_mid,
                region=ch_region,
                channel=ch_name
            )
            
            # Create trial-level dataframe
            # Average across amplitude frequencies to get (n_phase_freqs, n_trials)
            pac_avg_amp = z_scored_pac.mean(axis=0)  # (n_phase_freqs, n_trials)
            
            # Create dataframe with one row per trial
            trial_data = {
                'participant': subj_id,
                'trials': np.arange(1, min_trials + 1),  # 1-indexed to match learn_df
                'channel': ch_name,
                'region': ch_region,
            }
            
            # Add PAC metrics
            # 1. Average PAC across all frequency combinations (single value per trial)
            trial_data['pac_avg_all'] = z_scored_pac.mean(axis=(0, 1))[:min_trials]
            
            # 2. Average PAC across amplitude frequencies only (one value per phase freq per trial)
            # This gives us (n_trials, n_phase_freqs) - we'll store as separate columns
            for ph_idx, ph_freq in enumerate(f_pha_mid):
                trial_data[f'pac_ph{ph_idx:02d}_{ph_freq:.1f}Hz'] = pac_avg_amp[ph_idx, :min_trials]
            
            # 3. Sum of significant PAC values (|z| > 1.96, p < 0.05)
            z_scored_pac_sig = z_scored_pac.copy()
            z_scored_pac_sig[np.abs(z_scored_pac_sig) <= 1.96] = np.nan
            trial_data['pac_sum_sig'] = np.nansum(z_scored_pac_sig, axis=(0, 1))[:min_trials]
            
            # 4. Count of significant frequency pairs per trial
            trial_data['pac_n_sig'] = np.sum(np.abs(z_scored_pac) > 1.96, axis=(0, 1))[:min_trials]
            
            # Create dataframe
            elec_trial_df = pd.DataFrame(trial_data)
            
            # Merge with learn_df on participant and trials
            elec_trial_df = elec_trial_df.merge(
                subj_learn_df[['participant', 'trials'] + 
                             [col for col in subj_learn_df.columns if col not in ['participant', 'trials']]], 
                on=['participant', 'trials'], 
                how='left'
            )
            
            subj_trial_dfs.append(elec_trial_df)
            print(f"      Saved PAC data: {pac_save_file}")
            print(f"      Created trial dataframe: {len(elec_trial_df)} trials")
            
        except Exception as e:
            print(f"      Error processing {ch_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Save subject-level combined dataframe
    if len(subj_trial_dfs) > 0:
        subj_combined_df = pd.concat(subj_trial_dfs, ignore_index=True)
        subj_save_file = f'{save_dir_data}/{subj_id}_all_electrodes_pac_trials.csv'
        subj_combined_df.to_csv(subj_save_file, index=False)
        all_trial_dfs.append(subj_combined_df)
        print(f"  Saved subject-level dataframe: {subj_save_file} ({len(subj_combined_df)} rows)")
    
    # Clean up
    del epochs

# Combine all subjects into one dataframe
if len(all_trial_dfs) > 0:
    all_results_df = pd.concat(all_trial_dfs, ignore_index=True)
    combined_save_file = f'{save_dir_data}/all_subjects_pac_trials.csv'
    all_results_df.to_csv(combined_save_file, index=False)
    print(f"\n{'='*60}")
    print(f"COMPLETE: Saved combined results to {combined_save_file}")
    print(f"Total rows: {len(all_results_df)}")
    print(f"Subjects: {all_results_df.participant.nunique()}")
    print(f"Electrodes: {all_results_df.channel.nunique()}")
    print(f"Regions: {all_results_df.region.unique()}")
    print(f"{'='*60}")
else:
    print(f"\n{'='*60}")
    print("WARNING: No data was processed successfully!")
    print(f"{'='*60}")

In [ ]:
# Plot PAC by phase frequency split by rpe_category, separately for each ROI

# Load the combined results
combined_file = f'{save_dir_data}/all_subjects_pac_trials.csv'

if not os.path.exists(combined_file):
    print(f"Error: Combined results file not found: {combined_file}")
    print("Please run the previous cell first to generate the data.")
else:
    # Load data
    pac_df = pd.read_csv(combined_file)
    print(f"Loaded {len(pac_df)} rows from {combined_file}")
    
    # Check if rpe_category exists
    if 'rpe_category' not in pac_df.columns:
        print("\nWarning: 'rpe_category' column not found in dataframe.")
        print("Available columns:", pac_df.columns.tolist())
    elif 'region' not in pac_df.columns:
        print("\nWarning: 'region' column not found in dataframe.")
    else:
        # Filter out any missing rpe_category values
        pac_df_plot = pac_df[pac_df.rpe_category.notna()].copy()
        
        # Get phase frequency columns and extract frequencies
        phase_freq_cols = [col for col in pac_df_plot.columns if col.startswith('pac_ph')]
        phase_freqs = []
        for col in phase_freq_cols:
            # Extract frequency from column name like 'pac_ph00_4.0Hz'
            try:
                freq_part = col.split('_')[2]  # Get the frequency part
                freq = float(freq_part.replace('Hz', ''))
                phase_freqs.append(freq)
            except:
                continue
        phase_freqs = sorted(set(phase_freqs))
        
        # Get ROIs
        regions = sorted([r for r in pac_df_plot.region.unique() if r in rois])
        print(f"\nPlotting for regions: {regions}")
        
        # RPE categories and colors
        rpe_cats = ['negative', 'neutral', 'pos']
        colors = ['red', 'gray', 'green']
        
        # Create figure with one subplot per ROI
        n_regions = len(regions)
        fig, axes = plt.subplots(1, n_regions, figsize=(6*n_regions, 5))
        
        if n_regions == 1:
            axes = [axes]
        
        for region_idx, region in enumerate(regions):
            ax = axes[region_idx]
            
            # Filter data for this region
            region_data = pac_df_plot[pac_df_plot.region == region]
            
            # Plot each RPE category
            for rpe_cat, color in zip(rpe_cats, colors):
                cat_data = region_data[region_data.rpe_category == rpe_cat]
                if len(cat_data) > 0:
                    # Calculate mean and standard error for each phase frequency
                    pac_means = []
                    pac_sems = []
                    for ph_freq in phase_freqs:
                        # Find matching column
                        col_name = None
                        for col in phase_freq_cols:
                            if f'{ph_freq:.1f}Hz' in col or f'{ph_freq:.0f}Hz' in col:
                                col_name = col
                                break
                        if col_name and col_name in cat_data.columns:
                            values = cat_data[col_name].dropna()
                            if len(values) > 0:
                                mean_val = values.mean()
                                sem_val = values.std() / np.sqrt(len(values))  # Standard error
                                pac_means.append(mean_val)
                                pac_sems.append(sem_val)
                            else:
                                pac_means.append(np.nan)
                                pac_sems.append(np.nan)
                        else:
                            pac_means.append(np.nan)
                            pac_sems.append(np.nan)
                    
                    # Plot with error bars
                    ax.errorbar(phase_freqs, pac_means, yerr=pac_sems, 
                               label=rpe_cat, color=color, 
                               linewidth=2, markersize=6, alpha=0.8,
                               capsize=3, capthick=1.5, elinewidth=1.5)
            
            ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.3)
            ax.set_xlabel('Phase Frequency (Hz)', fontsize=12)
            ax.set_ylabel('Average PAC (z-scored)', fontsize=12)
            ax.set_title(f'{region}\nPAC by Phase Frequency (split by RPE Category)', 
                        fontsize=13, pad=15)
            ax.legend(loc='best', fontsize=11)
            ax.grid(True, alpha=0.3)
            ax.set_xlim([f_pha_start, f_pha_end])
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nPlot complete!")

In [ ]:
# Electrode-level statistics: Find electrodes with z-scores > 2 at any frequency for any RPE category

# Load the combined results
combined_file = f'{save_dir_data}/all_subjects_pac_trials.csv'

if not os.path.exists(combined_file):
    print(f"Error: Combined results file not found: {combined_file}")
    print("Please run the data processing cell first.")
else:
    # Load data
    pac_df = pd.read_csv(combined_file)
    print(f"Loaded {len(pac_df)} rows from {combined_file}")
    
    # Get phase frequency columns
    phase_freq_cols = [col for col in pac_df.columns if col.startswith('pac_ph')]
    print(f"Found {len(phase_freq_cols)} phase frequency columns")
    
    # Filter to rows with valid rpe_category
    pac_df_stats = pac_df[pac_df.rpe_category.notna()].copy()
    
    # Initialize results storage
    electrode_results = []
    
    # Group by participant, channel, and region
    for (participant, channel, region), group_df in pac_df_stats.groupby(['participant', 'channel', 'region']):
        electrode_info = {
            'participant': participant,
            'channel': channel,
            'region': region,
            'n_trials_total': len(group_df),
            'max_z_score': -np.inf,
            'max_z_freq': None,
            'max_z_rpe': None,
            'n_freqs_above_2': 0,
            'freqs_above_2': [],
            'rpe_cats_above_2': []
        }
        
        # Check each RPE category
        for rpe_cat in group_df.rpe_category.unique():
            cat_data = group_df[group_df.rpe_category == rpe_cat]
            
            # Check each phase frequency column
            for col in phase_freq_cols:
                if col in cat_data.columns:
                    values = cat_data[col].dropna()
                    if len(values) > 0:
                        # Get mean z-score for this frequency and RPE category
                        mean_z = values.mean()
                        
                        # Check if any individual trial exceeds 2
                        max_trial_z = values.max()
                        
                        # Track maximum
                        if mean_z > electrode_info['max_z_score']:
                            electrode_info['max_z_score'] = mean_z
                            electrode_info['max_z_freq'] = col
                            electrode_info['max_z_rpe'] = rpe_cat
                        
                        # Check if mean z-score > 2
                        if mean_z > 2.0:
                            electrode_info['n_freqs_above_2'] += 1
                            # Extract frequency from column name
                            try:
                                freq_part = col.split('_')[2].replace('Hz', '')
                                freq_val = float(freq_part)
                                electrode_info['freqs_above_2'].append(freq_val)
                                electrode_info['rpe_cats_above_2'].append(rpe_cat)
                            except:
                                electrode_info['freqs_above_2'].append(col)
                                electrode_info['rpe_cats_above_2'].append(rpe_cat)
        
        electrode_results.append(electrode_info)
    
    # Create results dataframe
    results_df = pd.DataFrame(electrode_results)
    
    # Filter to electrodes with z-scores > 2
    significant_electrodes = results_df[results_df['max_z_score'] > 2.0].copy()
    significant_electrodes = significant_electrodes.sort_values('max_z_score', ascending=False)
    
    print(f"\n{'='*80}")
    print("ELECTRODE-LEVEL STATISTICS")
    print(f"{'='*80}")
    print(f"\nTotal electrodes analyzed: {len(results_df)}")
    print(f"Electrodes with max z-score > 2.0: {len(significant_electrodes)}")
    
    if len(significant_electrodes) > 0:
        print(f"\n{'='*80}")
        print("ELECTRODES WITH Z-SCORES > 2.0")
        print(f"{'='*80}")
        
        # Display summary table
        display_cols = ['participant', 'channel', 'region', 'max_z_score', 'max_z_rpe', 
                       'max_z_freq', 'n_freqs_above_2', 'n_trials_total']
        print("\nSummary Table:")
        print(significant_electrodes[display_cols].to_string(index=False))
        
        # Detailed breakdown
        print(f"\n{'='*80}")
        print("DETAILED BREAKDOWN")
        print(f"{'='*80}")
        for idx, row in significant_electrodes.iterrows():
            print(f"\n{row['participant']} - {row['channel']} ({row['region']}):")
            print(f"  Max z-score: {row['max_z_score']:.3f}")
            print(f"  At frequency: {row['max_z_freq']}")
            print(f"  RPE category: {row['max_z_rpe']}")
            print(f"  Number of frequencies with mean z > 2: {row['n_freqs_above_2']}")
            if len(row['freqs_above_2']) > 0:
                print(f"  Frequencies above 2: {row['freqs_above_2']}")
                print(f"  RPE categories: {row['rpe_cats_above_2']}")
            print(f"  Total trials: {row['n_trials_total']}")
        
        # Summary by region
        print(f"\n{'='*80}")
        print("SUMMARY BY REGION")
        print(f"{'='*80}")
        region_summary = significant_electrodes.groupby('region').agg({
            'channel': 'count',
            'max_z_score': ['mean', 'max', 'min']
        }).round(3)
        region_summary.columns = ['n_electrodes', 'mean_max_z', 'max_z', 'min_max_z']
        print(region_summary)
        
        # Summary by RPE category
        print(f"\n{'='*80}")
        print("SUMMARY BY RPE CATEGORY (of max z-score)")
        print(f"{'='*80}")
        rpe_summary = significant_electrodes.groupby('max_z_rpe').agg({
            'channel': 'count',
            'max_z_score': ['mean', 'max']
        }).round(3)
        rpe_summary.columns = ['n_electrodes', 'mean_max_z', 'max_z']
        print(rpe_summary)
        
    else:
        print("\nNo electrodes found with z-scores > 2.0")
        print(f"Maximum z-score across all electrodes: {results_df['max_z_score'].max():.3f}")
    
    # Save results
    results_file = f'{save_dir_data}/electrode_level_statistics.csv'
    results_df.to_csv(results_file, index=False)
    print(f"\n{'='*80}")
    print(f"Results saved to: {results_file}")
    print(f"{'='*80}")

In [ ]:
# Plot results for each ROI
if len(all_subjects_data) > 0:
    phase_freqs = f_pha_mid
    n_regions = len(rois)
    
    # Create subplots: one per region
    fig, axes = plt.subplots(1, n_regions, figsize=(10 * n_regions, 6), sharey=True)
    if n_regions == 1:
        axes = [axes]
    
    for region_idx, region in enumerate(rois):
        ax = axes[region_idx]
        
        # Collect all PAC values across all subjects and electrodes for this region
        all_pac_means = []  # List of arrays, one per electrode
        all_pac_ses = []    # List of arrays, one per electrode
        
        # Collect data from all subjects
        for subj_id, subj_data in all_subjects_data.items():
            if region in subj_data:
                for elec_name, (pac_avg, pac_se) in subj_data[region].items():
                    all_pac_means.append(pac_avg)
                    all_pac_ses.append(pac_se)
        
        if len(all_pac_means) > 0:
            all_pac_means = np.array(all_pac_means)  # (n_elecs, n_phase_freqs)
            all_pac_ses = np.array(all_pac_ses)      # (n_elecs, n_phase_freqs)
            
            # Plot individual electrodes (transparent)
            for i in range(len(all_pac_means)):
                ax.plot(phase_freqs, all_pac_means[i], 'o-', linewidth=0.5, 
                       markersize=2, alpha=0.2, color='gray', label='_nolegend_')
            
            # Compute and plot group mean across all electrodes with error bars
            group_mean = all_pac_means.mean(axis=0)  # (n_phase_freqs,)
            
            # Standard error: combine within-electrode and across-electrode variance
            within_elec_var = (all_pac_ses ** 2).mean(axis=0)  # Mean of squared SEs
            across_elec_var = all_pac_means.var(axis=0)        # Variance of means
            group_se = np.sqrt(within_elec_var + across_elec_var) / np.sqrt(len(all_pac_means))
            
            # Plot group mean with error bars
            ax.errorbar(phase_freqs, group_mean, yerr=group_se,
                       fmt='o-', linewidth=3, markersize=8, color='steelblue',
                       capsize=5, capthick=2, elinewidth=2, 
                       label=f'Mean (n={len(all_pac_means)} electrodes)', zorder=10)
        
        ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        ax.set_xlabel('Phase Frequency (Hz)', fontsize=12)
        if region_idx == 0:
            ax.set_ylabel('Z-scored PAC (averaged across amplitude frequencies)', fontsize=12)
        ax.set_title(f'{region}\n(n={len(all_pac_means)} electrodes, {len([s for s in all_subjects_data.keys() if region in all_subjects_data[s]])} subjects)', 
                    fontsize=13, pad=15)
        ax.grid(True, alpha=0.3)
        ax.set_xlim([f_pha_start, f_pha_end])
        ax.legend(loc='best', fontsize=10)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot results for each ROI
if len(all_subjects_data) > 0:
    phase_freqs = f_pha_mid
    n_regions = len(rois)
    
    # Create subplots: one per region
    fig, axes = plt.subplots(1, n_regions, figsize=(10 * n_regions, 6), sharey=True)
    if n_regions == 1:
        axes = [axes]
    
    for region_idx, region in enumerate(rois):
        ax = axes[region_idx]
        
        # Collect all PAC values across all subjects and electrodes for this region
        all_pac_means = []  # List of arrays, one per electrode
        all_pac_ses = []    # List of arrays, one per electrode
        
        # Collect data from all subjects
        for subj_id, subj_data in all_subjects_data.items():
            if region in subj_data:
                for elec_name, (pac_avg, pac_se) in subj_data[region].items():
                    all_pac_means.append(pac_avg)
                    all_pac_ses.append(pac_se)
        
        if len(all_pac_means) > 0:
            all_pac_means = np.array(all_pac_means)  # (n_elecs, n_phase_freqs)
            all_pac_ses = np.array(all_pac_ses)      # (n_elecs, n_phase_freqs)
            
            # # Plot individual electrodes (transparent)
            # for i in range(len(all_pac_means)):
            #     ax.plot(phase_freqs, all_pac_means[i], 'o-', linewidth=0.5, 
            #            markersize=2, alpha=0.2, color='gray', label='_nolegend_')
            
            # Compute and plot group mean across all electrodes with error bars
            group_mean = all_pac_means.mean(axis=0)  # (n_phase_freqs,)
            
            # Standard error: combine within-electrode and across-electrode variance
            within_elec_var = (all_pac_ses ** 2).mean(axis=0)  # Mean of squared SEs
            across_elec_var = all_pac_means.var(axis=0)        # Variance of means
            group_se = np.sqrt(within_elec_var + across_elec_var) / np.sqrt(len(all_pac_means))
            
            # Plot group mean with error bars
            ax.errorbar(phase_freqs, group_mean, yerr=group_se,
                       fmt='o-', linewidth=3, markersize=8, color='steelblue',
                       capsize=5, capthick=2, elinewidth=2, 
                       label=f'Mean (n={len(all_pac_means)} electrodes)', zorder=10)
        
        ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        ax.set_xlabel('Phase Frequency (Hz)', fontsize=12)
        if region_idx == 0:
            ax.set_ylabel('Z-scored PAC (averaged across amplitude frequencies)', fontsize=12)
        ax.set_title(f'{region}\n(n={len(all_pac_means)} electrodes, {len([s for s in all_subjects_data.keys() if region in all_subjects_data[s]])} subjects)', 
                    fontsize=13, pad=15)
        ax.grid(True, alpha=0.3)
        ax.set_xlim([f_pha_start, f_pha_end])
        ax.legend(loc='best', fontsize=10)
    
    plt.tight_layout()
    plt.show()

In [ ]:
surr_mean.shape

In [ ]:
z_scored_pac.max()

## 5. Statistical Testing: ERPAC Permutation Tests

In [ ]:
# Load epochs to get time information for ERPAC
# We'll use the first subject's epochs as reference for time axis
sample_epoch_file = None
for subj_id in subj_ids:
    save_path = f'{ephys_dir}/{subj_id}/neural/{day}'
    epoch_file = f'{save_path}/{event}-epo.fif'
    if os.path.exists(epoch_file):
        sample_epochs = mne.read_epochs(epoch_file, preload=False, verbose=False)
        times = sample_epochs.times
        sfreq = sample_epochs.info['sfreq']
        del sample_epochs
        break

print(f"Time axis: {times[0]:.3f} to {times[-1]:.3f} seconds ({len(times)} timepoints)")
print(f"Sampling frequency: {sfreq} Hz")

In [ ]:
# Perform permutation testing for ERPAC
# Test 1: One-sample test against 0 for each RPE category
# Test 2: Compare ERPAC between RPE categories

from mne.stats import permutation_cluster_test
import pickle

erpac_cluster_results = []

# Group by subject and electrode
for (subj_id, ch_name), group in tqdm(results_df.groupby(['subject', 'electrode']), 
                                      desc="ERPAC cluster testing"):
    ch_region = group['region'].iloc[0]
    
    # Load ERPAC data for all categories for this electrode
    erpac_data = {}
    categories_present = []
    
    for rpe_cat in ['negative', 'neutral', 'pos']:
        erpac_file = f'{save_dir_data}/{subj_id}_{ch_name}_erpac_{rpe_cat}.npz'
        if os.path.exists(erpac_file):
            data = np.load(erpac_file)
            erpac_data[rpe_cat] = data['z_scored_erpac']  # (n_amp, n_pha, n_times)
            categories_present.append(rpe_cat)
    
    if len(categories_present) < 2:
        continue  # Need at least 2 categories to compare
    
    # Test 1: One-sample test against 0 for each category
    for rpe_cat in categories_present:
        erpac = erpac_data[rpe_cat]
        
        # Average across frequency bands for time-resolved test
        # Option 1: Average across all frequency pairs
        erpac_mean = erpac.mean(axis=(0, 1))  # (n_times,)
        
        # Reshape for cluster test: (n_observations, n_times)
        # For ERPAC, we have one observation per timepoint (averaged across trials)
        # So we test if the mean ERPAC at each timepoint is significantly > 0
        # We'll use the full 3D array and test across time dimension
        erpac_for_test = erpac.reshape(-1, erpac.shape[-1])  # (n_amp*n_pha, n_times)
        
        # One-sample test against 0
        t_obs, clusters, cluster_pvals, h0 = permutation_cluster_1samp_test(
            erpac_for_test,
            n_permutations=n_perm_cluster,
            n_jobs=-1,
            tail=1,  # One-tailed (positive ERPAC)
            verbose=False
        )
        
        has_sig_cluster = any(cluster_pvals <= cluster_alpha) if len(cluster_pvals) > 0 else False
        min_p = np.min(cluster_pvals) if len(cluster_pvals) > 0 else 1.0
        
        erpac_cluster_results.append({
            'subject': subj_id,
            'electrode': ch_name,
            'region': ch_region,
            'rpe_category': rpe_cat,
            'test_type': 'one_sample',
            'has_significant_cluster': has_sig_cluster,
            'min_cluster_p': min_p,
            'n_clusters': len(clusters)
        })
        
        # Save cluster test results
        # Note: clusters is a list of boolean arrays with inhomogeneous shapes, so save separately with pickle
        np.savez_compressed(
            f'{save_dir_data}/{subj_id}_{ch_name}_erpac_{rpe_cat}_cluster_stats.npz',
            t_obs=t_obs,
            cluster_pvals=cluster_pvals,
            times=times,
            test_type='one_sample'
        )
        # Save clusters separately using pickle
        with open(f'{save_dir_data}/{subj_id}_{ch_name}_erpac_{rpe_cat}_clusters.pkl', 'wb') as f:
            pickle.dump(clusters, f)
    
    # Test 2: Compare between categories (if we have multiple categories)
    if len(categories_present) >= 2:
        # Compare negative vs neutral
        if 'negative' in categories_present and 'neutral' in categories_present:
            erpac_neg = erpac_data['negative'].reshape(-1, erpac_data['negative'].shape[-1])
            erpac_neu = erpac_data['neutral'].reshape(-1, erpac_data['neutral'].shape[-1])
            
            # Two-sample test
            t_obs, clusters, cluster_pvals, h0 = permutation_cluster_test(
                [erpac_neg, erpac_neu],
                n_permutations=n_perm_cluster,
                n_jobs=-1,
                tail=0,  # Two-tailed
                verbose=False
            )
            
            has_sig_cluster = any(cluster_pvals <= cluster_alpha) if len(cluster_pvals) > 0 else False
            min_p = np.min(cluster_pvals) if len(cluster_pvals) > 0 else 1.0
            
            erpac_cluster_results.append({
                'subject': subj_id,
                'electrode': ch_name,
                'region': ch_region,
                'rpe_category': 'negative_vs_neutral',
                'test_type': 'two_sample',
                'has_significant_cluster': has_sig_cluster,
                'min_cluster_p': min_p,
                'n_clusters': len(clusters)
            })
            
            np.savez_compressed(
                f'{save_dir_data}/{subj_id}_{ch_name}_erpac_neg_vs_neu_cluster_stats.npz',
                t_obs=t_obs,
                cluster_pvals=cluster_pvals,
                times=times,
                test_type='two_sample'
            )
            # Save clusters separately using pickle
            with open(f'{save_dir_data}/{subj_id}_{ch_name}_erpac_neg_vs_neu_clusters.pkl', 'wb') as f:
                pickle.dump(clusters, f)
        
        # Compare negative vs positive
        if 'negative' in categories_present and 'pos' in categories_present:
            erpac_neg = erpac_data['negative'].reshape(-1, erpac_data['negative'].shape[-1])
            erpac_pos = erpac_data['pos'].reshape(-1, erpac_data['pos'].shape[-1])
            
            t_obs, clusters, cluster_pvals, h0 = permutation_cluster_test(
                [erpac_neg, erpac_pos],
                n_permutations=n_perm_cluster,
                n_jobs=-1,
                tail=0,
                verbose=False
            )
            
            has_sig_cluster = any(cluster_pvals <= cluster_alpha) if len(cluster_pvals) > 0 else False
            min_p = np.min(cluster_pvals) if len(cluster_pvals) > 0 else 1.0
            
            erpac_cluster_results.append({
                'subject': subj_id,
                'electrode': ch_name,
                'region': ch_region,
                'rpe_category': 'negative_vs_pos',
                'test_type': 'two_sample',
                'has_significant_cluster': has_sig_cluster,
                'min_cluster_p': min_p,
                'n_clusters': len(clusters)
            })
            
            np.savez_compressed(
                f'{save_dir_data}/{subj_id}_{ch_name}_erpac_neg_vs_pos_cluster_stats.npz',
                t_obs=t_obs,
                cluster_pvals=cluster_pvals,
                times=times,
                test_type='two_sample'
            )
            # Save clusters separately using pickle
            with open(f'{save_dir_data}/{subj_id}_{ch_name}_erpac_neg_vs_pos_clusters.pkl', 'wb') as f:
                pickle.dump(clusters, f)
        
        # Compare neutral vs positive
        if 'neutral' in categories_present and 'pos' in categories_present:
            erpac_neu = erpac_data['neutral'].reshape(-1, erpac_data['neutral'].shape[-1])
            erpac_pos = erpac_data['pos'].reshape(-1, erpac_data['pos'].shape[-1])
            
            t_obs, clusters, cluster_pvals, h0 = permutation_cluster_test(
                [erpac_neu, erpac_pos],
                n_permutations=n_perm_cluster,
                n_jobs=-1,
                tail=0,
                verbose=False
            )
            
            has_sig_cluster = any(cluster_pvals <= cluster_alpha) if len(cluster_pvals) > 0 else False
            min_p = np.min(cluster_pvals) if len(cluster_pvals) > 0 else 1.0
            
            erpac_cluster_results.append({
                'subject': subj_id,
                'electrode': ch_name,
                'region': ch_region,
                'rpe_category': 'neutral_vs_pos',
                'test_type': 'two_sample',
                'has_significant_cluster': has_sig_cluster,
                'min_cluster_p': min_p,
                'n_clusters': len(clusters)
            })
            
            np.savez_compressed(
                f'{save_dir_data}/{subj_id}_{ch_name}_erpac_neu_vs_pos_cluster_stats.npz',
                t_obs=t_obs,
                cluster_pvals=cluster_pvals,
                times=times,
                test_type='two_sample'
            )
            # Save clusters separately using pickle
            with open(f'{save_dir_data}/{subj_id}_{ch_name}_erpac_neu_vs_pos_clusters.pkl', 'wb') as f:
                pickle.dump(clusters, f)

# Create cluster results dataframe
erpac_cluster_df = pd.DataFrame(erpac_cluster_results)
if len(erpac_cluster_df) > 0:
    erpac_cluster_df.to_csv(f'{save_dir}/erpac_cluster_results.csv', index=False)
    
    print(f"\nERPAC cluster testing complete")
    print(f"Total tests: {len(erpac_cluster_df)}")
    print(f"\nSignificant one-sample tests by region:")
    one_sample = erpac_cluster_df[erpac_cluster_df.test_type == 'one_sample']
    if len(one_sample) > 0:
        print(one_sample.groupby(['region', 'rpe_category'])['has_significant_cluster'].sum())
    print(f"\nSignificant two-sample tests by region:")
    two_sample = erpac_cluster_df[erpac_cluster_df.test_type == 'two_sample']
    if len(two_sample) > 0:
        print(two_sample.groupby(['region', 'rpe_category'])['has_significant_cluster'].sum())
else:
    print("No cluster test results generated")

## 6. Visualization: Time-Resolved ERPAC Plots

In [ ]:
def plot_time_resolved_erpac(erpac, times, t_obs=None, cluster_pvals=None, clusters=None,
                             title='', xlabel='Time (s)', ylabel='Mean ERPAC (z-scored)',
                             save_path=None, cluster_alpha=0.05):
    """
    Plot time-resolved ERPAC with optional cluster-based significance masking.
    
    Parameters
    ----------
    erpac : array
        ERPAC values averaged across frequency bands, shape (n_times,)
    times : array
        Time axis in seconds
    t_obs : array, optional
        T-statistics from cluster test, shape (n_times,)
    cluster_pvals : array, optional
        P-values for each cluster
    clusters : list, optional
        Cluster masks
    title : str
        Plot title
    save_path : str, optional
        Path to save figure
    """
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Plot ERPAC time course
    ax.plot(times, erpac, 'k-', linewidth=2, label='ERPAC')
    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    
    # Highlight significant clusters if provided
    if t_obs is not None and clusters is not None and cluster_pvals is not None:
        for cl, pval in zip(clusters, cluster_pvals):
            if pval <= cluster_alpha:
                # Get timepoints in this cluster
                cl_times = times[cl]
                cl_erpac = erpac[cl]
                ax.fill_between(cl_times, 0, cl_erpac, alpha=0.3, color='red',
                               label=f'Significant cluster (p={pval:.3f})')
    
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    sns.despine()
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    return fig


def plot_erpac_comparison(erpac_dict, times, cluster_stats_dict=None, 
                          title='', save_path=None, cluster_alpha=0.05):
    """
    Plot ERPAC time courses for different RPE categories with comparisons.
    
    Parameters
    ----------
    erpac_dict : dict
        Dictionary with keys as category names and values as ERPAC arrays (n_times,)
    times : array
        Time axis in seconds
    cluster_stats_dict : dict, optional
        Dictionary with cluster test results for comparisons
    title : str
        Plot title
    save_path : str, optional
        Path to save figure
    """
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    # Color scheme for RPE categories
    colors = {'negative': '#E74C3C', 'neutral': '#95A5A6', 'pos': '#3498DB'}
    
    # Top panel: Individual category time courses
    ax1 = axes[0]
    for cat, erpac in erpac_dict.items():
        color = colors.get(cat, 'gray')
        ax1.plot(times, erpac, color=color, linewidth=2, label=cat.capitalize(), alpha=0.8)
    
    ax1.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax1.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax1.set_ylabel('Mean ERPAC (z-scored)', fontsize=12)
    ax1.set_title(f'{title} - Individual Categories', fontsize=13, fontweight='bold')
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    sns.despine(ax=ax1)
    
    # Bottom panel: Differences between categories
    ax2 = axes[1]
    if cluster_stats_dict is not None:
        for comp_name, stats in cluster_stats_dict.items():
            if 't_obs' in stats and 'clusters' in stats:
                t_obs = stats['t_obs']
                clusters = stats['clusters']
                cluster_pvals = stats.get('cluster_pvals', [])
                
                # Average t-statistics across frequency dimensions if needed
                if t_obs.ndim > 1:
                    t_obs_mean = t_obs.mean(axis=0) if t_obs.ndim == 2 else t_obs.mean(axis=(0, 1))
                else:
                    t_obs_mean = t_obs
                
                ax2.plot(times, t_obs_mean, linewidth=2, label=comp_name.replace('_', ' ').title(), alpha=0.7)
                
                # Highlight significant clusters
                if len(cluster_pvals) > 0:
                    for cl, pval in zip(clusters, cluster_pvals):
                        if pval <= cluster_alpha:
                            cl_times = times[cl]
                            cl_t = t_obs_mean[cl] if t_obs_mean.ndim == 0 else t_obs_mean[cl]
                            ax2.fill_between(cl_times, 0, cl_t, alpha=0.3, color='red')
    
    ax2.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax2.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax2.set_xlabel('Time (s)', fontsize=12)
    ax2.set_ylabel('T-statistic (difference)', fontsize=12)
    ax2.set_title('Category Comparisons', fontsize=13, fontweight='bold')
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    sns.despine(ax=ax2)
    
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    return fig

In [ ]:
# Generate plots for electrodes with significant ERPAC
import pickle  # Ensure pickle is imported for loading clusters

if len(erpac_cluster_df) > 0:
    # Get electrodes with significant one-sample tests
    sig_electrodes = erpac_cluster_df[
        (erpac_cluster_df.test_type == 'one_sample') & 
        (erpac_cluster_df.has_significant_cluster == True)
    ][['subject', 'electrode', 'region']].drop_duplicates()
    
    print(f"Generating plots for {len(sig_electrodes)} electrodes with significant ERPAC...")
    
    for idx, row in tqdm(sig_electrodes.iterrows(), total=len(sig_electrodes), desc="Plotting"):
        subj_id = row['subject']
        ch_name = row['electrode']
        ch_region = row['region']
        
        # Load ERPAC data for all categories
        erpac_dict = {}
        cluster_stats_dict = {}
        
        for rpe_cat in ['negative', 'neutral', 'pos']:
            erpac_file = f'{save_dir_data}/{subj_id}_{ch_name}_erpac_{rpe_cat}.npz'
            if os.path.exists(erpac_file):
                data = np.load(erpac_file)
                erpac = data['z_scored_erpac']  # (n_amp, n_pha, n_times)
                # Average across frequency bands
                erpac_mean = erpac.mean(axis=(0, 1))  # (n_times,)
                erpac_dict[rpe_cat] = erpac_mean
                
                # Load cluster stats if available
                cluster_file = f'{save_dir_data}/{subj_id}_{ch_name}_erpac_{rpe_cat}_cluster_stats.npz'
                cluster_pkl_file = f'{save_dir_data}/{subj_id}_{ch_name}_erpac_{rpe_cat}_clusters.pkl'
                if os.path.exists(cluster_file):
                    cluster_data = np.load(cluster_file)
                    # Load clusters from pickle file
                    clusters = None
                    if os.path.exists(cluster_pkl_file):
                        with open(cluster_pkl_file, 'rb') as f:
                            clusters = pickle.load(f)
                    cluster_stats_dict[f'{rpe_cat}_one_sample'] = {
                        't_obs': cluster_data['t_obs'],
                        'clusters': clusters,
                        'cluster_pvals': cluster_data['cluster_pvals']
                    }
        
        # Load comparison cluster stats
        for comp in ['neg_vs_neu', 'neg_vs_pos', 'neu_vs_pos']:
            comp_file = f'{save_dir_data}/{subj_id}_{ch_name}_erpac_{comp}_cluster_stats.npz'
            comp_pkl_file = f'{save_dir_data}/{subj_id}_{ch_name}_erpac_{comp}_clusters.pkl'
            if os.path.exists(comp_file):
                comp_data = np.load(comp_file)
                # Load clusters from pickle file
                clusters = None
                if os.path.exists(comp_pkl_file):
                    with open(comp_pkl_file, 'rb') as f:
                        clusters = pickle.load(f)
                cluster_stats_dict[comp] = {
                    't_obs': comp_data['t_obs'],
                    'clusters': clusters,
                    'cluster_pvals': comp_data['cluster_pvals']
                }
        
        if len(erpac_dict) > 0:
            # Reconstruct times from ERPAC data shape (accounting for edge removal)
            # Load epoch file to get original times and sampling frequency
            save_path = f'{ephys_dir}/{subj_id}/neural/{day}'
            epoch_file = f'{save_path}/{event}-epo.fif'
            if os.path.exists(epoch_file):
                epochs = mne.read_epochs(epoch_file, preload=False, verbose=False)
                sfreq = epochs.info['sfreq']
                original_times = epochs.times
                edges = int(buf_ms * (sfreq / 1000))  # Convert buffer to samples
                del epochs
                
                # Get actual time dimension from ERPAC data
                first_erpac = list(erpac_dict.values())[0]
                n_times_erpac = len(first_erpac)
                
                # Compute times after edge removal
                # ERPAC has shape (n_amp, n_pha, n_times) where n_times = original_n_times - 2*edges
                # So we need to slice the original times
                if n_times_erpac < len(original_times):
                    # Account for edge removal
                    times_erpac = original_times[edges:edges+n_times_erpac]
                else:
                    times_erpac = original_times[:n_times_erpac]
            else:
                # Fallback: create times from ERPAC shape assuming same sampling rate
                first_erpac = list(erpac_dict.values())[0]
                n_times_erpac = len(first_erpac)
                # Use a default time range if we can't load epochs
                times_erpac = np.linspace(-1.0, 2.0, n_times_erpac)  # Default: -1 to 2 seconds
            
            # Plot comparison
            plot_erpac_comparison(
                erpac_dict, times_erpac, cluster_stats_dict,
                title=f'{subj_id} - {ch_name} ({ch_region})',
                save_path=f'{save_dir_plots}/{subj_id}_{ch_name}_erpac_comparison.pdf'
            )
            
            # Plot individual category with significance
            for rpe_cat, erpac_mean in erpac_dict.items():
                cluster_key = f'{rpe_cat}_one_sample'
                stats = cluster_stats_dict.get(cluster_key, {})
                
                t_obs = stats.get('t_obs', None)
                clusters = stats.get('clusters', None)
                cluster_pvals = stats.get('cluster_pvals', None)
                
                if t_obs is not None:
                    # Average t_obs if multi-dimensional
                    if t_obs.ndim > 1:
                        t_obs_mean = t_obs.mean(axis=0) if t_obs.ndim == 2 else t_obs.mean(axis=(0, 1))
                    else:
                        t_obs_mean = t_obs
                else:
                    t_obs_mean = None
                
                plot_time_resolved_erpac(
                    erpac_mean, times_erpac, t_obs_mean, cluster_pvals, clusters,
                    title=f'{subj_id} - {ch_name} ({ch_region}) - {rpe_cat.capitalize()}',
                    save_path=f'{save_dir_plots}/{subj_id}_{ch_name}_erpac_{rpe_cat}.pdf'
                )
    
    print(f"Plots saved to {save_dir_plots}")
else:
    print("No significant electrodes found for plotting")

In [ ]:
# Create region-level summary plots
# Aggregate ERPAC across electrodes within each region

region_erpac_summary = {}

# Get times from first available ERPAC file (accounting for edge removal)
times_erpac_region = None
for roi in rois:
    roi_electrodes = results_df[results_df.region == roi]
    if len(roi_electrodes) > 0:
        # Get first electrode to determine time axis
        first_row = roi_electrodes.iloc[0]
        subj_id = first_row['subject']
        ch_name = first_row['electrode']
        
        # Try to get times from epoch file
        save_path = f'{ephys_dir}/{subj_id}/neural/{day}'
        epoch_file = f'{save_path}/{event}-epo.fif'
        if os.path.exists(epoch_file):
            epochs = mne.read_epochs(epoch_file, preload=False, verbose=False)
            sfreq = epochs.info['sfreq']
            original_times = epochs.times
            edges = int(buf_ms * (sfreq / 1000))
            del epochs
            
            # Get time dimension from ERPAC
            for rpe_cat in ['negative', 'neutral', 'pos']:
                erpac_file = f'{save_dir_data}/{subj_id}_{ch_name}_erpac_{rpe_cat}.npz'
                if os.path.exists(erpac_file):
                    data = np.load(erpac_file)
                    erpac = data['z_scored_erpac']
                    n_times_erpac = erpac.shape[-1]
                    times_erpac_region = original_times[edges:edges+n_times_erpac]
                    break
            if times_erpac_region is not None:
                break

# If we couldn't get times, use a default
if times_erpac_region is None:
    # Get shape from first available ERPAC file
    for roi in rois:
        roi_electrodes = results_df[results_df.region == roi]
        if len(roi_electrodes) > 0:
            first_row = roi_electrodes.iloc[0]
            subj_id = first_row['subject']
            ch_name = first_row['electrode']
            for rpe_cat in ['negative', 'neutral', 'pos']:
                erpac_file = f'{save_dir_data}/{subj_id}_{ch_name}_erpac_{rpe_cat}.npz'
                if os.path.exists(erpac_file):
                    data = np.load(erpac_file)
                    erpac = data['z_scored_erpac']
                    n_times_erpac = erpac.shape[-1]
                    times_erpac_region = np.linspace(-1.0, 2.0, n_times_erpac)
                    break
            if times_erpac_region is not None:
                break

for roi in rois:
    roi_electrodes = results_df[results_df.region == roi]
    
    if len(roi_electrodes) == 0:
        continue
    
    region_erpac_summary[roi] = {}
    
    for rpe_cat in ['negative', 'neutral', 'pos']:
        erpac_list = []
        
        for idx, row in roi_electrodes.iterrows():
            subj_id = row['subject']
            ch_name = row['electrode']
            
            erpac_file = f'{save_dir_data}/{subj_id}_{ch_name}_erpac_{rpe_cat}.npz'
            if os.path.exists(erpac_file):
                data = np.load(erpac_file)
                erpac = data['z_scored_erpac']  # (n_amp, n_pha, n_times)
                erpac_mean = erpac.mean(axis=(0, 1))  # (n_times,)
                erpac_list.append(erpac_mean)
        
        if len(erpac_list) > 0:
            # Stack and compute mean and SEM
            erpac_stack = np.stack(erpac_list, axis=0)  # (n_electrodes, n_times)
            region_erpac_summary[roi][rpe_cat] = {
                'mean': erpac_stack.mean(axis=0),
                'sem': erpac_stack.std(axis=0) / np.sqrt(len(erpac_list)),
                'n_electrodes': len(erpac_list)
            }

# Plot region-level summaries
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

colors = {'negative': '#E74C3C', 'neutral': '#95A5A6', 'pos': '#3498DB'}

for i, roi in enumerate(rois):
    if roi not in region_erpac_summary:
        continue
    
    ax = axes[i]
    
    for rpe_cat in ['negative', 'neutral', 'pos']:
        if rpe_cat in region_erpac_summary[roi]:
            data = region_erpac_summary[roi][rpe_cat]
            mean = data['mean']
            sem = data['sem']
            n_elec = data['n_electrodes']
            
            color = colors[rpe_cat]
            ax.plot(times_erpac_region, mean, color=color, linewidth=2, label=rpe_cat.capitalize(), alpha=0.8)
            ax.fill_between(times_erpac_region, mean - sem, mean + sem, color=color, alpha=0.2)
    
    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Time (s)', fontsize=10)
    ax.set_ylabel('Mean ERPAC (z-scored)', fontsize=10)
    if times_erpac_region is not None:
        ax.set_xlim(times_erpac_region[0], times_erpac_region[-1])
    ax.set_title(f'{roi}\n(n={region_erpac_summary[roi].get("negative", {}).get("n_electrodes", 0)} electrodes)', 
                 fontsize=11, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    sns.despine(ax=ax)

plt.suptitle('Region-Level ERPAC Summary by RPE Category', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{save_dir_plots}/region_level_erpac_summary.pdf', dpi=300, bbox_inches='tight')
plt.show()

print(f"Region-level summary plot saved to {save_dir_plots}/region_level_erpac_summary.pdf")

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(all_results)
print(f"\nPAC computed for {len(results_df)} electrodes")
print(results_df.region.value_counts())

## 5. Statistical Testing: Cluster Permutation Tests


In [ ]:
# Perform cluster permutation testing for each electrode
cluster_results = []

for idx, row in tqdm(results_df.iterrows(), total=len(results_df), desc="Cluster testing"):
    subj_id = row['subject']
    ch_name = row['electrode']
    ch_region = row['region']
    
    # Load z-scored PAC
    pac_file = f'{save_dir_data}/{subj_id}_{ch_name}_pac.npz'
    data = np.load(pac_file)
    z_scored_pac = data['z_scored_pac']
    
    # Reshape for cluster test: (n_trials, n_amp, n_pha)
    pac_for_test = np.moveaxis(z_scored_pac, -1, 0)
    
    # Run cluster permutation test (one-sample against 0)
    t_obs, clusters, cluster_pvals, h0 = permutation_cluster_1samp_test(
        pac_for_test,
        n_permutations=n_perm_cluster,
        n_jobs=-1,
        tail=1,  # One-tailed (positive PAC)
        verbose=False
    )
    
    # Determine significance
    has_sig_cluster = any(cluster_pvals <= cluster_alpha) if len(cluster_pvals) > 0 else False
    min_p = np.min(cluster_pvals) if len(cluster_pvals) > 0 else 1.0
    
    # Get extent of significant clusters
    sig_cluster_extent = 0
    if has_sig_cluster:
        for cl, pval in zip(clusters, cluster_pvals):
            if pval <= cluster_alpha:
                sig_cluster_extent += np.sum(cl)
    
    cluster_results.append({
        'subject': subj_id,
        'electrode': ch_name,
        'region': ch_region,
        'has_significant_cluster': has_sig_cluster,
        'min_cluster_p': min_p,
        'cluster_extent': sig_cluster_extent,
        'n_clusters': len(clusters),
        'hemisphere': row['hemisphere']
    })
    
    # Save cluster test results
    np.savez_compressed(
        f'{save_dir_data}/{subj_id}_{ch_name}_cluster_stats.npz',
        t_obs=t_obs,
        cluster_pvals=cluster_pvals,
        n_clusters=len(clusters)
    )

# Create cluster results dataframe
cluster_df = pd.DataFrame(cluster_results)
cluster_df.to_csv(f'{save_dir}/pac_results.csv', index=False)

print(f"\nCluster testing complete")
print(f"Significant electrodes by region:")
print(cluster_df.groupby('region')['has_significant_cluster'].sum())


## 6. Visualization: Cluster-Masked Comodulograms


In [ ]:
def plot_cluster_masked_comodulogram(z_pac, t_obs, cluster_pvals, clusters, 
                                      pac_obj, title='', save_path=None):
    """
    Create a publication-quality cluster-masked comodulogram.
    
    Parameters
    ----------
    z_pac : array
        Mean z-scored PAC across trials (n_amp, n_pha)
    t_obs : array
        T-statistics from cluster test
    cluster_pvals : array
        P-values for each cluster
    clusters : list
        Cluster masks
    pac_obj : Pac
        Tensorpac PAC object for plotting
    title : str
        Plot title
    save_path : str, optional
        Path to save figure
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Left panel: Raw z-scored PAC
    plt.sca(axes[0])
    pac_obj.comodulogram(
        z_pac.mean(axis=-1) if z_pac.ndim == 3 else z_pac,
        title='Z-scored PAC',
        cmap='Spectral_r',
        vmin=-3, vmax=3
    )
    axes[0].set_xlabel('Phase Frequency (Hz)')
    axes[0].set_ylabel('Amplitude Frequency (Hz)')
    
    # Right panel: Cluster-masked
    plt.sca(axes[1])
    
    # Create masked version
    t_obs_sig = np.nan * np.ones_like(t_obs)
    t_obs_nonsig = t_obs.copy()
    
    if len(cluster_pvals) > 0:
        for cl, pval in zip(clusters, cluster_pvals):
            if pval <= cluster_alpha:
                t_obs_sig[cl] = t_obs[cl]
                t_obs_nonsig[cl] = np.nan
    
    # Plot non-significant in gray
    pac_obj.comodulogram(
        t_obs_nonsig,
        cmap='gray',
        vmin=0, vmax=np.nanmax(t_obs) if not np.all(np.isnan(t_obs)) else 1,
        colorbar=False
    )
    
    # Plot significant clusters in color
    if not np.all(np.isnan(t_obs_sig)):
        pac_obj.comodulogram(
            t_obs_sig,
            title=f'Significant Clusters (p < {cluster_alpha})',
            cmap='hot',
            vmin=0, vmax=np.nanmax(t_obs)
        )
    else:
        axes[1].set_title('No Significant Clusters')
    
    axes[1].set_xlabel('Phase Frequency (Hz)')
    axes[1].set_ylabel('Amplitude Frequency (Hz)')
    
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    return fig


In [ ]:
# Generate plots for significant electrodes
sig_electrodes = cluster_df[cluster_df.has_significant_cluster]

print(f"Generating plots for {len(sig_electrodes)} significant electrodes...")

for idx, row in tqdm(sig_electrodes.iterrows(), total=len(sig_electrodes)):
    subj_id = row['subject']
    ch_name = row['electrode']
    ch_region = row['region']
    
    # Load PAC and cluster stats
    pac_data = np.load(f'{save_dir_data}/{subj_id}_{ch_name}_pac.npz')
    cluster_data = np.load(f'{save_dir_data}/{subj_id}_{ch_name}_cluster_stats.npz', 
                          allow_pickle=True)
    
    z_pac = pac_data['z_scored_pac']
    t_obs = cluster_data['t_obs']
    cluster_pvals = cluster_data['cluster_pvals']
    
    # Reconstruct clusters from t_obs for visualization
    # (We need to re-run to get cluster masks, but we have t_obs)
    pac_for_test = np.moveaxis(z_pac, -1, 0)
    _, clusters, _, _ = permutation_cluster_1samp_test(
        pac_for_test, n_permutations=100, n_jobs=-1, tail=1, verbose=False
    )
    
    plot_cluster_masked_comodulogram(
        z_pac, t_obs, cluster_pvals, clusters, pac_obj,
        title=f'{subj_id} - {ch_name} ({ch_region})',
        save_path=f'{save_dir_plots}/{subj_id}_{ch_name}_comodulogram.pdf'
    )

print(f"Individual electrode plots saved to {save_dir_plots}")


In [ ]:
pac_data.keys()

## 7. Summary Statistics and Region Comparisons


In [ ]:
# Aggregate PAC across electrodes by region
region_pac = {roi: [] for roi in rois}
region_t_obs = {roi: [] for roi in rois}

for idx, row in cluster_df.iterrows():
    subj_id = row['subject']
    ch_name = row['electrode']
    ch_region = row['region']
    
    pac_data = np.load(f'{save_dir_data}/{subj_id}_{ch_name}_pac.npz')
    cluster_data = np.load(f'{save_dir_data}/{subj_id}_{ch_name}_cluster_stats.npz')
    
    # Average across trials
    mean_pac = pac_data['z_scored_pac'].mean(axis=-1)
    region_pac[ch_region].append(mean_pac)
    region_t_obs[ch_region].append(cluster_data['t_obs'])

# Compute grand averages
for roi in rois:
    if len(region_pac[roi]) > 0:
        region_pac[roi] = np.stack(region_pac[roi]).mean(axis=0)
        region_t_obs[roi] = np.stack(region_t_obs[roi]).mean(axis=0)


In [ ]:
# Create region summary plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, roi in enumerate(rois):
    if isinstance(region_pac[roi], np.ndarray):
        plt.sca(axes[i])
        pac_obj.comodulogram(
            region_pac[roi],
            title=f'{roi} - Mean Z-scored PAC\n(n={len(cluster_df[cluster_df.region == roi])} electrodes)',
            cmap='Spectral_r',
            vmin=-0.1, vmax=0.1
        )
        axes[i].set_xlabel('Phase Frequency (Hz)')
        axes[i].set_ylabel('Amplitude Frequency (Hz)')

plt.tight_layout()
plt.savefig(f'{save_dir_plots}/ACC_dmPFC_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Save individual region summary plots
for roi in rois:
    if isinstance(region_pac[roi], np.ndarray):
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        plt.sca(ax)
        
        n_elec = len(cluster_df[cluster_df.region == roi])
        n_sig = cluster_df[(cluster_df.region == roi) & (cluster_df.has_significant_cluster)].shape[0]
        
        pac_obj.comodulogram(
            region_pac[roi],
            title=f'{roi} Mean Z-scored PAC\n{n_sig}/{n_elec} electrodes with significant clusters',
            cmap='Spectral_r',
            vmin=-2, vmax=2
        )
        ax.set_xlabel('Phase Frequency (Hz)', fontsize=12)
        ax.set_ylabel('Amplitude Frequency (Hz)', fontsize=12)
        
        plt.tight_layout()
        plt.savefig(f'{save_dir_plots}/{roi}_summary.pdf', dpi=300, bbox_inches='tight')
        plt.close()


In [ ]:
# Summary statistics table
print("\n" + "="*60)
print("PAC ANALYSIS SUMMARY")
print("="*60)

for roi in rois:
    roi_data = cluster_df[cluster_df.region == roi]
    n_total = len(roi_data)
    n_sig = roi_data.has_significant_cluster.sum()
    pct_sig = 100 * n_sig / n_total if n_total > 0 else 0
    
    print(f"\n{roi}:")
    print(f"  Total electrodes: {n_total}")
    print(f"  Significant electrodes: {n_sig} ({pct_sig:.1f}%)")
    if n_sig > 0:
        print(f"  Mean cluster extent (sig only): {roi_data[roi_data.has_significant_cluster].cluster_extent.mean():.1f}")

print("\n" + "="*60)
print(f"Results saved to: {save_dir}/pac_results.csv")
print(f"Plots saved to: {save_dir_plots}")
print("="*60)


In [ ]:
# Create bar plot of significant electrodes by region
from scipy.stats import binomtest

sig_counts = cluster_df.groupby('region')['has_significant_cluster'].agg(['sum', 'count'])
sig_counts['pct'] = 100 * sig_counts['sum'] / sig_counts['count']

# Compute binomial confidence intervals
ci_low, ci_high, pvals = [], [], []
for roi in rois:
    if roi in sig_counts.index:
        result = binomtest(int(sig_counts.loc[roi, 'sum']), 
                          int(sig_counts.loc[roi, 'count']), 
                          p=0.05)
        lb, ub = result.proportion_ci(confidence_level=0.95, method='wilson')
        ci_low.append(lb * 100)
        ci_high.append(ub * 100)
        pvals.append(result.pvalue)

sig_counts = sig_counts.loc[rois]
sig_counts['ci_low'] = ci_low
sig_counts['ci_high'] = ci_high
sig_counts['pval'] = pvals

# Plot
fig, ax = plt.subplots(figsize=(6, 5))

colors = ['#E74C3C', '#3498DB']  # Red for ACC, Blue for dmPFC
bars = ax.bar(rois, sig_counts['pct'], 
              yerr=[sig_counts['pct'] - sig_counts['ci_low'],
                    sig_counts['ci_high'] - sig_counts['pct']],
              capsize=5, color=colors, edgecolor='black', linewidth=1.5)

# Add significance markers
for i, (roi, pval) in enumerate(zip(rois, pvals)):
    if pval < 0.001:
        marker = '***'
    elif pval < 0.01:
        marker = '**'
    elif pval < 0.05:
        marker = '*'
    else:
        marker = ''
    
    if marker:
        ax.text(i, sig_counts.loc[roi, 'ci_high'] + 3, marker,
               ha='center', va='bottom', fontsize=16, fontweight='bold')

# Reference line at 5%
ax.axhline(5, color='gray', linestyle='--', linewidth=1.5, label='Chance (5%)')

ax.set_xlabel('Brain Region', fontsize=12)
ax.set_ylabel('% Electrodes with Significant PAC', fontsize=12)
ax.set_title('Significant Phase-Amplitude Coupling\nby Brain Region', fontsize=14)
ax.legend(loc='upper right')
ax.set_ylim(0, max(sig_counts['ci_high']) + 15)

sns.despine()
plt.tight_layout()
plt.savefig(f'{save_dir_plots}/significance_barplot.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("\nStatistical summary:")
print(sig_counts[['sum', 'count', 'pct', 'pval']].rename(
    columns={'sum': 'n_sig', 'count': 'n_total', 'pct': 'pct_sig', 'pval': 'binom_p'}))


In [ ]:
## 8. Region-Level Cluster Permutation Tests

# Aggregate PAC data across electrodes within each region and perform cluster permutation testing at the region level.

In [ ]:
# Aggregate PAC data by region for region-level permutation testing
# For each region, we stack all electrode PAC data and test across electrodes

region_level_results = {}

for roi in rois:
    print(f"\n{'='*50}")
    print(f"Region-level cluster test for {roi}")
    print('='*50)
    
    # Get all electrodes in this region
    roi_electrodes = results_df[results_df.region == roi]
    
    if len(roi_electrodes) == 0:
        print(f"No electrodes found in {roi}")
        continue
    
    # Collect PAC data from all electrodes in this region
    all_electrode_pac = []
    
    for idx, row in roi_electrodes.iterrows():
        subj_id = row['subject']
        ch_name = row['electrode']
        
        pac_file = f'{save_dir_data}/{subj_id}_{ch_name}_pac.npz'
        data = np.load(pac_file)
        z_scored_pac = data['z_scored_pac']
        
        # Average across trials for this electrode: (n_amp, n_pha)
        mean_electrode_pac = z_scored_pac.mean(axis=-1)
        all_electrode_pac.append(mean_electrode_pac)
    
    # Stack all electrodes: (n_electrodes, n_amp, n_pha)
    stacked_pac = np.stack(all_electrode_pac, axis=0)
    
    print(f"Number of electrodes: {stacked_pac.shape[0]}")
    print(f"PAC shape per electrode: {stacked_pac.shape[1:]}")
    
    # Perform cluster permutation test across electrodes
    # Testing if the mean PAC across electrodes is significantly > 0
    t_obs, clusters, cluster_pvals, h0 = permutation_cluster_1samp_test(
        stacked_pac,
        n_permutations=n_perm_cluster,
        threshold=1,
        n_jobs=-1,
        tail=1,  # One-tailed (positive PAC)
        verbose=False
    )
    
    # Determine significance
    has_sig_cluster = any(cluster_pvals <= cluster_alpha) if len(cluster_pvals) > 0 else False
    min_p = np.min(cluster_pvals) if len(cluster_pvals) > 0 else 1.0
    n_sig_clusters = np.sum(cluster_pvals <= cluster_alpha) if len(cluster_pvals) > 0 else 0
    
    print(f"Number of clusters found: {len(clusters)}")
    print(f"Significant clusters (p < {cluster_alpha}): {n_sig_clusters}")
    print(f"Minimum cluster p-value: {min_p:.4f}")
    
    # Store results
    region_level_results[roi] = {
        'stacked_pac': stacked_pac,
        't_obs': t_obs,
        'clusters': clusters,
        'cluster_pvals': cluster_pvals,
        'has_sig_cluster': has_sig_cluster,
        'min_p': min_p,
        'n_electrodes': stacked_pac.shape[0]
    }
    
    # Save region-level results
    np.savez_compressed(
        f'{save_dir_data}/{roi}_region_cluster_stats.npz',
        t_obs=t_obs,
        cluster_pvals=cluster_pvals,
        mean_pac=stacked_pac.mean(axis=0),
        n_electrodes=stacked_pac.shape[0]
    )


In [ ]:
# Visualize region-level cluster-masked comodulograms
fig, axes = plt.subplots(4, 2, figsize=(14, 24))

for i, roi in enumerate(rois):
    if roi not in region_level_results:
        continue
    
    res = region_level_results[roi]
    t_obs = res['t_obs']
    clusters = res['clusters']
    cluster_pvals = res['cluster_pvals']
    mean_pac = res['stacked_pac'].mean(axis=0)
    
    # Left column: Raw mean z-scored PAC
    ax_raw = axes[i, 0]
    plt.sca(ax_raw)
    pac_obj.comodulogram(
        mean_pac,
        title=f'{roi} Mean Z-scored PAC\n(n={res["n_electrodes"]} electrodes)',
        cmap='Spectral_r',
        vmin=0, vmax=0.1
    )
    ax_raw.set_xlabel('Phase Frequency (Hz)')
    ax_raw.set_ylabel('Amplitude Frequency (Hz)')
    
    # Right column: Cluster-masked t-statistics
    ax_mask = axes[i, 1]
    plt.sca(ax_mask)
    
    # Create masked version
    t_obs_sig = np.nan * np.ones_like(t_obs)
    t_obs_nonsig = t_obs.copy()
    
    if len(cluster_pvals) > 0:
        for cl, pval in zip(clusters, cluster_pvals):
            if pval <= cluster_alpha:
                t_obs_sig[cl] = t_obs[cl]
                t_obs_nonsig[cl] = np.nan
    
    # Plot non-significant in gray
    pac_obj.comodulogram(
        t_obs_nonsig,
        cmap='gray',
        vmin=0, vmax=np.nanmax(t_obs) if not np.all(np.isnan(t_obs)) else 1,
        colorbar=False
    )
    
    # Plot significant clusters in color
    if not np.all(np.isnan(t_obs_sig)):
        pac_obj.comodulogram(
            t_obs_sig,
            title=f'{roi} Significant Clusters (p < {cluster_alpha})\nMin p = {res["min_p"]:.4f}',
            cmap='hot',
            vmin=0, vmax=np.nanmax(t_obs)
        )
        # invert the y-axis
        ax_mask.invert_yaxis()
    else:
        ax_mask.set_title(f'{roi} No Significant Clusters\nMin p = {res["min_p"]:.4f}')
    
    ax_mask.set_xlabel('Phase Frequency (Hz)')
    ax_mask.set_ylabel('Amplitude Frequency (Hz)')

plt.tight_layout()
plt.savefig(f'{save_dir_plots}/region_level_cluster_test.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("\nRegion-level cluster test visualization saved!")


In [ ]:
# Summary of region-level cluster permutation tests
print("\n" + "="*70)
print("REGION-LEVEL CLUSTER PERMUTATION TEST SUMMARY")
print("="*70)

summary_data = []
for roi in rois:
    if roi in region_level_results:
        res = region_level_results[roi]
        sig_status = "SIGNIFICANT" if res['has_sig_cluster'] else "Not significant"
        
        # Count significant clusters and their total extent
        sig_cluster_extent = 0
        if res['has_sig_cluster']:
            for cl, pval in zip(res['clusters'], res['cluster_pvals']):
                if pval <= cluster_alpha:
                    sig_cluster_extent += np.sum(cl)
        
        print(f"\n{roi}:")
        print(f"  Number of electrodes: {res['n_electrodes']}")
        print(f"  Total clusters found: {len(res['clusters'])}")
        print(f"  Significant clusters: {np.sum(res['cluster_pvals'] <= cluster_alpha) if len(res['cluster_pvals']) > 0 else 0}")
        print(f"  Minimum cluster p-value: {res['min_p']:.4f}")
        print(f"  Significant cluster extent: {sig_cluster_extent} frequency bins")
        print(f"  Status: {sig_status}")
        
        summary_data.append({
            'Region': roi,
            'N_electrodes': res['n_electrodes'],
            'N_clusters': len(res['clusters']),
            'N_sig_clusters': np.sum(res['cluster_pvals'] <= cluster_alpha) if len(res['cluster_pvals']) > 0 else 0,
            'Min_p': res['min_p'],
            'Sig_extent': sig_cluster_extent,
            'Significant': res['has_sig_cluster']
        })

# Create summary dataframe
region_summary_df = pd.DataFrame(summary_data)
region_summary_df.to_csv(f'{save_dir}/region_level_results.csv', index=False)

print("\n" + "="*70)
print(f"Results saved to: {save_dir}/region_level_results.csv")
print("="*70)

# Display summary table
display(region_summary_df)


In [ ]:
day = 'Day1'

rois = ['HPC', 'OFC', 'dmPFC', 'AMY', 'dlPFC', 'ACC']

for subj_id in subj_ids:
    all_dfs = [] 

    event = 'feedback_start'
    baseline_event = 'baseline_start'
    # baseline_start_mem
    loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
    save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'

    # For each subject, load the epochs of interest and the baseline epoch, as well as the electrode dataframe 
    epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif',  preload=True) 

    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')

    buf_ms = 1000 # buffer in ms
    edges = buf_ms * (epochs_reref.info['sfreq']/1000) # convert ms to samples

    # compute true pac and surrogates
    elecs = np.where(elec_df.salman_region.isin(rois))[0] 

    pwise_dfs = []
    if len(elecs)>0:
        for ei in elecs:
            ch_name = elec_df.loc[ei, 'label']
            ch_region = elec_df.loc[ei, 'salman_region']

            real_sig = np.squeeze(epochs_reref.copy().pick(ch_name)._data)

            epoch_pac = p.filterfit(epochs_reref.info['sfreq'], 
                                real_sig, 
                                n_perm=n_perm, n_jobs=-1,
                                edges=edges, random_state=0)


            surr_mean = p.surrogates.mean(axis=0)
            surr_std = p.surrogates.std(axis=0)
            z_scored_pac = (epoch_pac-surr_mean)/surr_std

            # Save out the z_scored pac so I can do individual analysis on it
            np.save(f'{save_dir_data}/{subj_id}_{ch_name}_beta_pac', z_scored_pac)
            
            # such as: t_obs, clusters, cluster_p_values, h0 = permutation_cluster_1samp_test(np.moveaxis(z_scored_pac, -1, 0), 
                                                                    # n_permutations=1000, n_jobs=-1, tail=1)

            # average the z-scores across phae and frequency amplitudes 
            
#             avg_conn = z_scored_pac.mean(axis=(0, 1))
            
#             # alternative: nan out non-significant pac_values, then sum
#             z_scored_pac[np.abs(z_scored_pac)<=1.96] = np.nan
            
#             sum_conn = np.nansum(z_scored_pac, axis=(0, 1))

#             pwise_df = pd.DataFrame(columns=['participant', 'roi', 'label', 'event', 
#                                                 'avg_conn', 'sum_conn', 'hemi'])

#             pwise_df['avg_conn'] = avg_conn
#             pwise_df['sum_conn'] = sum_conn
#             pwise_df['hemi'] = elec_df[elec_df.label==ch_name].hemisphere.values[0]
#             pwise_df['participant'] = subj_id
#             pwise_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
#             pwise_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]

#             pwise_df['region'] = ch_region
#             pwise_df['event'] = event
#             pwise_df['band0'] = phase_frequency
#             pwise_df['band1'] = amp_frequency
#             pwise_df['trials'] = np.arange(1, len(avg_conn)+1)
#             pwise_df['label'] = ch_name


#             pwise_df = pwise_df.merge(learn_df[learn_df.participant==subj_id], on=['participant', 'trials'])

#             pwise_df['zrpe'] = (pwise_df.rpe - np.nanmean(pwise_df.rpe)) / (2*np.nanstd(pwise_df.rpe))
#             pwise_df['zpm'] = (pwise_df.DPRIME - np.nanmean(pwise_df.DPRIME)) / (2*np.nanstd(pwise_df.DPRIME))

# #             # aggregate
#             pwise_dfs.append(pwise_df)
#     else:
#         # this subject does not have electrodes in both regions
#         continue
    
#     if len(pwise_dfs) > 0:
#         all_df_pt = pd.concat(pwise_dfs)
#         all_df_pt.to_csv(f'{save_dir_data}/{subj_id}_pac_{event}_5_4.csv', index=False)


In [ ]:
save_dir_plots = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-PAC'
save_dir_data = f'{base_dir}/work/qasims01/MemoryBanditData/EMU/dataframes'
rois = ['HPC', 'OFC', 'dmPFC', 'AMY', 'dlPFC', 'ACC']

# 
n_perm = 500  # number of permutations
# define the Pac object (gaussian-copula PAC) 
p = Pac(idpac=(6, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')

day = 'Day1'

all_dfs = [] 
sig_elecs = []

for subj_id in subj_ids:

    event = 'feedback_start'
    baseline_event = 'baseline_start'
    # baseline_start_mem
    loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
    save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'

    # For each subject, load the epochs of interest and the baseline epoch, as well as the electrode dataframe 
    epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif',  preload=True) 

    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
    ch_names = epochs_reref.ch_names
    del epochs_reref

    elecs = np.where(elec_df.salman_region.isin(rois))[0]     
     
    pwise_dfs = []
    if len(elecs)>0:
        for ei in elecs:
            ch_name = elec_df.loc[ei, 'label']
            ch_region = elec_df.loc[ei, 'salman_region']

            z_scored_pac = np.load(f'{save_dir_data}/{subj_id}_{ch_name}_theta_pac.npy')
            
            t_obs, clusters, cluster_p_values, h0 = permutation_cluster_1samp_test(np.moveaxis(z_scored_pac, -1, 0), 
                                                                   n_permutations=1000, n_jobs=-1, tail=1)
            if any(cluster_p_values<=0.05):
                # print('sig electrode')
                # sig_elecs.append(subj_id)
                f = plt.figure(figsize=(11, 5))
                plt.subplot(121)
                p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')
                plt.savefig(f'{save_dir_plots}/{subj_id}_{ch_name}_theta_pac.pdf', dpi=300)
                plt.close(f)
                significant_pac = 1
            else:
                significant_pac = 0

            # Get just the theta pac only
            # z_scored_pac_theta = z_scored_pac[:, np.where(f_pha[:, 1]<9)[0], :]
            z_scored_pac_theta = z_scored_pac

            avg_conn_theta = z_scored_pac_theta.mean(axis=(0, 1))

            # # Get the pac for alpha (greater or equal to 9 and less than 13
            # z_scored_pac_alpha = z_scored_pac[:, np.where((f_pha[:, 0]>=7) & (f_pha[:, 0]<=10))[0], :]
            # avg_conn_alpha = z_scored_pac_alpha.mean(axis=(0, 1))

            # # Get the pac for beta (greater or equal to 13 and less than 30
            # z_scored_pac_beta = z_scored_pac[:, np.where(f_pha[:, 1]>=16)[0], :]
            # avg_conn_beta = z_scored_pac_beta.mean(axis=(0, 1))

            
            # # alternative: nan out non-significant pac_values, then sum
            # z_scored_pac[np.abs(z_scored_pac)<=1.96] = np.nan
            # z_scored_pac_theta[np.abs(z_scored_pac_theta)<=1.96] = np.nan
            # z_scored_pac_alpha[np.abs(z_scored_pac_alpha)<=1.96] = np.nan
            # z_scored_pac_beta[np.abs(z_scored_pac_beta)<=1.96] = np.nan

            # sum_conn_theta = np.nansum(z_scored_pac_theta, axis=(0, 1))
            # sum_conn_alpha = np.nansum(z_scored_pac_alpha, axis=(0, 1))
            # sum_conn_beta = np.nansum(z_scored_pac_beta, axis=(0, 1))

            pwise_df = pd.DataFrame(columns=['participant', 'roi', 'label', 'event', 
                                                 'hemi'])


            pwise_df['avg_conn_theta'] = avg_conn_theta
            # pwise_df['avg_conn_alpha'] = avg_conn_alpha
            # pwise_df['avg_conn_beta'] = avg_conn_beta
            # pwise_df['sum_conn_theta'] = sum_conn_theta
            # pwise_df['sum_conn_alpha'] = sum_conn_alpha
            # pwise_df['sum_conn_beta'] = sum_conn_beta
            # pwise_df['sum_conn'] = sum_conn
            pwise_df['hemi'] = elec_df[elec_df.label==ch_name].hemisphere.values[0]
            pwise_df['participant'] = subj_id
            pwise_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            pwise_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]

            pwise_df['region'] = ch_region
            pwise_df['event'] = event
            pwise_df['band0'] = phase_frequency
            pwise_df['band1'] = amp_frequency
            pwise_df['trials'] = np.arange(1, len(avg_conn_theta)+1)
            pwise_df['label'] = ch_name


            pwise_df = pwise_df.merge(learn_df[learn_df.participant==subj_id], on=['participant', 'trials'])

            pwise_df['zrpe'] = (pwise_df.rpe - np.nanmean(pwise_df.rpe)) / (2*np.nanstd(pwise_df.rpe))
            pwise_df['zpm'] = (pwise_df.DPRIME - np.nanmean(pwise_df.DPRIME)) / (2*np.nanstd(pwise_df.DPRIME))

            # rtrials = pwise_df.loc[pwise_df.reward==1, 'trials'].values - 1
            # nrtrials = pwise_df.loc[pwise_df.reward==0, 'trials'].values - 1
            # rpac = z_scored_pac[:, :, rtrials]
            # nrpac = z_scored_pac[:, :, nrtrials]

            # t_obs, clusters, cluster_p_values, h0 = permutation_cluster_test([np.moveaxis(rpac, -1, 0),
            #                                                         np.moveaxis(nrpac, -1, 0)], 
            #                                                         n_permutations=1000, n_jobs=-1, tail=1)
            
            # if any(cluster_p_values<=0.05):
            #     # print('sig electrode')
            #     # sig_elecs.append(subj_id)
            #     # f = plt.figure(figsize=(11, 5))
            #     # plt.subplot(121)
            #     # p.comodulogram(t_obs, title=str(p), cmap='Spectral_r')
            #     significant_pac = 1
            # else:
            #     significant_pac = 0

            pwise_df['significant_pac'] = significant_pac

# #             # aggregate
            pwise_dfs.append(pwise_df)
    else:
        # this subject does not have electrodes in both regions
        continue
    
    if len(pwise_dfs) > 0:
        all_df_pt = pd.concat(pwise_dfs)
        all_df_pt.to_csv(f'{save_dir_data}/{subj_id}_pac_{event}_theta.csv', index=False)
